# Marketplace Listing Integrity Agent

A hybrid pipeline that decides whether a marketplace listing's image genuinely matches its text description — grounded by a **RAG** knowledge base of category-specific verification policy, with an **LLM** reserved for the small number of steps that are genuine judgment calls, not every step.

### Why this isn't a full ReAct loop end to end
The first version of this agent put every step behind an LLM decision — detect category, detect product, decide to retrieve a policy, decide to check a feature, decide when to stop. Testing surfaced two problems: it took ~1-2 minutes per listing locally (8-12 sequential LLM turns, each re-processing a growing context), and most of those turns weren't actually judgment calls — "call detect_category, then detect_product" is the same fixed sequence every time. Worse, making OCR/VQA fallback an LLM *choice* meant the model sometimes skipped it entirely, "inferring" an unconfirmed brand from context instead of checking — exactly the failure mode this project exists to catch.

This version keeps the LLM only where testing showed it's genuinely needed. Everything else is deterministic Python, the same way the sibling ridesharing project's own Stage 1 handled its rule-based logic.

### Pipeline at a glance

| Stage | What happens | Needs an LLM? |
|---|---|---|
| 1. Detect category, then product | two short constrained VQA questions | No — fixed order, no judgment |
| 2. Self-consistency check + retry | does the product belong to the category? | No — a membership check + a fixed one-time retry rule |
| 3. Retrieve category policy | RAG: exact match, semantic search as fallback | No — embedding model only |
| 4. **Extract description features** | map raw listing text onto the resolved category's known feature schema | **Yes — genuine language understanding, the extraction step below** |
| 5. Hard-stop vs. description | does the image-grounded product agree with the extracted claim? | No — normalized string comparison |
| 6. Build checklist | policy's critical features + any other concrete claim | No — set logic |
| 7. Verify each feature | OCR first, then a fixed-template VQA fallback question | No — deterministic matching |
| 8. Resolve anything still ambiguous | **only if step 7 left something unresolved** | **Yes — narrow escalation, rare** |
| 9. Final verdict | hard veto on any critical mismatch | No — computed, not generated |

A listing where every feature resolves cleanly in step 7 makes exactly one LLM call in the whole pipeline — the extraction step. Everything else, including category/product detection and every comparison, is deterministic.

### Why extraction needed an LLM when nothing else in this pipeline did
Every other step turned out to be reliably handled by code once tested — category detection, checklist building, feature comparison. Extraction is different: real listings arrive as raw text a seller wrote (`"Prairie Farms Whole Milk, 1 Quart Carton"`), not a pre-structured dict, and mapping arbitrary phrasing onto known feature names (recognizing `"Whole"` describes the `variant` slot, not `product_type`) requires real language understanding a keyword rule can't approximate at real marketplace scale. Testing this took three iterations — see Step 4 below.

> **Requires:** a local [Ollama](https://ollama.com) server running `qwen2.5:14b` and `nomic-embed-text` (embeddings). See the project README for setup.

## Setup

In [ ]:
import sys
import os
import time
sys.path.append(os.path.abspath(".."))

from src.config import OLLAMA_MODEL, EMBED_MODEL
from src.policy_corpus import CATEGORY_POLICIES
from src.retrieval import build_policy_corpus, retrieve_category_policy, get_checklist_features, get_bounded_vocabulary
from src.extraction import extract_description_features
from src.tools import ALL_TOOLS
from src.vision_tools import detect_category_and_product
from src.agent import build_graph, verify_listing, _build_checklist, _deterministic_verify
from src.listings import LISTINGS

print(f"Reasoning model : {OLLAMA_MODEL}  (used for extraction every listing, escalation only if needed)")
print(f"Embedding model : {EMBED_MODEL}")
print(f"LLM-facing tools: {[t.name for t in ALL_TOOLS]}  (escalation sub-graph only)")

---
## Step 1: Building the Category Policy Knowledge Base (RAG)

Unchanged from the full-ReAct version — each category still gets a policy document with critical/cosmetic features, embedded into Chroma. What's different: `retrieve_category_policy` is now a plain function the pipeline always calls once category is known, not a tool an LLM decides to call.

In [ ]:
vectorstore = build_policy_corpus()
print(f"Collection size: {len(vectorstore.get()['ids'])} documents")

for category in ["grocery", "a pair of wireless bluetooth headphones", "a cotton crew-neck t-shirt"]:
    text = retrieve_category_policy(category)
    resolved = text.split("Category:")[1].split("(")[0].strip().rstrip(".")
    print(f"'{category}' -> {resolved}")

---
## Step 2: Deterministic Category & Product Detection

`detect_category_and_product` runs both VQA questions, checks whether the answers are self-consistent, and retries once if not — all in plain code. No LLM reasoning turn happens here; there's nothing to reason about, just two tool calls and a membership check.

In [ ]:
listing = LISTINGS[0]
print(f"Listing: {listing['name']}\n")

t0 = time.time()
category, product, product_confirmed = detect_category_and_product(listing["image_url"])
print(f"category           : {category}")
print(f"product            : {product}")
print(f"product_confirmed  : {product_confirmed}")
print(f"(took {time.time() - t0:.1f}s, zero LLM reasoning calls)")

---
## Step 4: Extracting Structured Features from Raw Description Text

This is the one genuine LLM-judgment step in the whole pipeline. `extract_description_features` maps the listing's raw text onto the category's known feature schema (from Step 1's RAG lookup) — open text extraction for unbounded fields (`brand`, `size`), a closed-vocabulary pick for fields with a genuinely bounded real-world vocabulary (`variant`, `container`).

**Why the closed-vocabulary split matters:** open extraction alone was tested and found unreliable specifically for `variant` — asking the model to freely extract and split `"Whole Milk"` into `product_type: "milk"` + `variant: "whole"` either merged them into one field or dropped `variant` entirely, even with full field descriptions and few-shot examples. Constraining just that field to a closest-match pick from a known list (same principle as `detect_category`/`detect_product`'s closed option lists) fixed it immediately. `brand` and `size` have no realistic bounded vocabulary, so they stay open text.

In [ ]:
critical_features, cosmetic_features = get_checklist_features(category)
bounded_vocab = get_bounded_vocabulary(category)
print(f"Known schema for '{category}': {critical_features + cosmetic_features}")
print(f"Bounded-vocabulary fields: {list(bounded_vocab.keys())}\n")

t0 = time.time()
description_features = extract_description_features(
    listing["description"], category, critical_features, cosmetic_features,
)
print(f"Raw text  : {listing['description']!r}")
print(f"Extracted : {description_features}")
print(f"(took {time.time() - t0:.1f}s, one LLM call)")

---
## Step 5: Deterministic Feature Verification

For each checklist feature: try OCR first (`check_against_ocr`), fall back to a fixed-template VQA question if OCR doesn't find it (`check_against_vqa_answer`). Both comparisons are normalized string matching — same mechanism that fixed the plural/singular and "guessing at unstructured OCR text" bugs found during testing, now applied deterministically instead of hoping a prompt instruction gets followed.

In [ ]:
checklist, checklist_critical = _build_checklist(critical_features, description_features)
print(f"Checklist: {checklist}\n")

t0 = time.time()
results, ambiguous = _deterministic_verify(listing["image_url"], checklist)
for feature, r in results.items():
    status = "match" if r["match"] is True else "mismatch" if r["match"] is False else "unconfirmed"
    print(f"  {feature:<10} [{r['source']:<3}] {status:<12} (claimed: '{r['value']}')")
print(f"\nAmbiguous (needs LLM escalation): {ambiguous or 'none'}")
print(f"(took {time.time() - t0:.1f}s)")

---
## Step 6: The Narrow Escalation Sub-Graph

Standard LangGraph ReAct shape (`agent` <-> `tools` via `tools_condition`) — but bound to only `read_label_text` and `ask_vision_question`, and only ever invoked when Step 3 leaves something ambiguous. This is the only part of the pipeline that's still genuinely agentic, and it's reached far less often than every step was in the full-ReAct version.

In [ ]:
graph = build_graph()

try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("Diagram rendering unavailable, printing graph structure instead:")
    print(graph.get_graph().draw_mermaid())

---
## Step 7: Running the Full Pipeline

Now the whole thing end to end via `verify_listing`, across all 5 listings — a correct one, three with injected mismatches (wrong brand, wrong size, wrong product entirely), and one more correct listing as a control. Watch the `escalated` field and the timing: every listing now makes exactly one LLM call (extraction) unless something is genuinely ambiguous, versus the full-ReAct version's ~8-12 sequential calls.

In [ ]:
for listing in LISTINGS:
    print(f"Listing: {listing['name']}")
    print(f"Description: {listing['description']!r}")
    t0 = time.time()
    result = verify_listing(graph, listing["image_url"], listing["description"])
    elapsed = time.time() - t0

    print(f"  extracted   : {result['description_features']}")
    print(f"  category    : {result['category']}")
    print(f"  hard_stop   : {result['hard_stop']}")
    print(f"  escalated   : {result['escalated']}  (ambiguous features: {result['ambiguous_features'] or 'none'})")
    print(f"  took        : {elapsed:.1f}s")
    print()
    print(result["answer"])
    print("\n" + "=" * 70 + "\n")

---
## Summary

### What changed from the full-ReAct version
The pipeline used to put every decision behind an LLM call — ~8-12 sequential reasoning turns per listing, most of them not actually judgment calls. This version:

- **Runs category/product detection, the self-consistency check, policy retrieval, the hard-stop comparison, and checklist building entirely as plain code** — no LLM involved, because none of these steps were ever genuinely ambiguous
- **Extracts structured features from raw listing text with exactly one LLM call** — the one step that turned out to genuinely need language understanding, not orchestration
- **Verifies each feature deterministically** — OCR first, a fixed-template VQA question as fallback, normalized string matching for both
- **Escalates to a narrow LLM ReAct loop only for features that stay genuinely ambiguous** after the deterministic pass — most listings never reach this step
- **Computes the verdict and writes the explanation from code**, not generation

### Key findings
- The specific bug that let a wrong-brand listing pass as "Likely match" — the LLM skipping `ask_vision_question` and inferring an answer instead — is structurally impossible now: OCR-then-VQA-fallback always runs in code
- The plural/singular false-mismatch bug is fixed by normalization applied uniformly, not by a prompt rule a smaller model could ignore
- **Extraction needed real iteration to get right**: a first attempt with a bare schema (no field descriptions) extracted only 2 of 5 fields correctly; adding field descriptions alone changed nothing (same 2 fields succeeded, same 3 failed — ruling out "insufficient hints" as the cause); adding few-shot examples got 4 of 5, with the last failure (`"Whole Milk"` not splitting into `product_type`/`variant`) traced to open-ended extraction being unreliable for a field with a genuinely bounded vocabulary; making `variant` a closed-vocabulary pick instead of open text fixed the last gap
- Escalation frequency itself is a useful signal — a category/listing type that escalates often is telling you its `feature_questions` or `common_products` list needs work

### Limitations
- **The hard-stop and category/product consistency checks are deterministic with no escalation path** — a genuinely ambiguous disagreement is resolved by a fixed rule, not investigated further
- **Bounded-vocabulary lists (`variant`, `container`) are hand-curated and grocery-only** — electronics/apparel don't have one yet, so their `variant`-equivalent fields (if any) would extract as open text, untested
- **Extraction is only validated on grocery listings** — same category-scope gap as the rest of the project
- **No human-in-the-loop yet** — every verdict is fully automatic

### Potential future work
- Bounded vocabularies for electronics/apparel's own bounded-ish fields (e.g. `color`)
- Measure escalation rate across a larger, more varied test set
- A human-in-the-loop gate before a "Likely mismatch" verdict executes, for high-risk categories or sellers
- Real verified images for electronics and apparel to close the live-testing gap